In [47]:
import pandas as pd
import numpy as np
import itertools
import random
import joblib
import os

random.seed(42)
np.random.seed(42)

data = pd.read_csv("jopacc_system_month_november_2022_to_june_2026.csv")

In [22]:
system_features = {

    "JoMoPay": [
        "wallet_individual_count",
        "wallet_commercial_count",
        "tx_cash_in_count",
        "tx_cash_out_count"
    ],
    "CliQ": [
        "customer_individual_count",
        "customer_legal_entity_count",
        "tx_purchases_count",
        "tx_money_transfers_count"
    ],
    "eFAWATEERcom": [
        "user_new_count",
        "billers_total",
        "ef_cash_payment_count",
        "ef_digital_payment_count"
    ],
    "ACH": [
        "ach_jod_transaction_count",
        "ach_usd_transaction_count",
        "ach_eur_transaction_count",
        "ach_gbp_transaction_count"
    ],
    "ECCU": [
        "transactions_total",
        "returned_cheque_count",
        "returned_cheque_count_rate_pct",
        "returned_cheque_value_rate_pct"
    ]
}

In [23]:
system_data = {}
system_bins = {}

for system in system_features:

    system_df = data[data["system"] == system].copy() # Select rows of the current system
    system_bins[system] = {} # store the bin boundaries
    features = system_features[system]  # features of the current system

    for feature in features:

        level_column = feature + "_level"
        system_df[level_column], bins = pd.qcut(
            system_df[feature],3,
            labels=[0, 1, 2],
            retbins=True,
            duplicates="drop"
        )
        system_df[level_column] = system_df[level_column].astype(int)
        system_bins[system][feature] = bins

    # state from the four feature levels
    level_columns = [feature + "_level" for feature in features]
    system_df["State"] = system_df[level_columns].apply(tuple,axis=1)

    # current system data
    system_data[system] = system_df

In [24]:
# ACTIONS AND COSTS
system_actions = {
    "JoMoPay": {
        "Do Nothing": 0,
        "Promote Wallet Usage": 35,
        "Encourage Wallet Cash-In": 25,
        "Reduce Cash-Out Dependency": 35
    },
    "CliQ": {
        "Do Nothing": 0,
        "Increase Customer Participation": 35,
        "Promote CliQ Purchases": 25,
        "Encourage Money Transfers": 20
    },
    "eFAWATEERcom": {
        "Do Nothing": 0,
        "Attract New Users": 25,
        "Onboard New Billers": 35,
        "Move Cash Payments to Digital Channels": 30
    },
    "ACH": {
        "Do Nothing": 0,
        "Promote JOD Transfers": 25,
        "Promote USD Transfers": 30,
        "Promote EUR and GBP Transfers": 35
    },
    "ECCU": {
        "Do Nothing": 0,
        "Improve Cheque Processing": 25,
        "Reduce Returned-Cheque Count": 35,
        "Reduce Returned-Cheque Risk Rates": 40
    }
}

In [25]:
# STATE TRANSITION

def apply_action(system, state, action):

    # Convert the tuple to a list so it can be changed
    next_state = list(state)

    if action == "Do Nothing":
        return tuple(next_state)

    if system == "JoMoPay":

        if action == "Promote Wallet Usage":
            next_state[0] = min(next_state[0] + 1, 2)
            next_state[1] = min(next_state[1] + 1, 2)

        elif action == "Encourage Wallet Cash-In":
            next_state[2] = min(next_state[2] + 1, 2)

        elif action == "Reduce Cash-Out Dependency":
            next_state[3] = max(next_state[3] - 1, 0)

    elif system == "CliQ":

        if action == "Increase Customer Participation":
            next_state[0] = min(next_state[0] + 1, 2)
            next_state[1] = min(next_state[1] + 1, 2)

        elif action == "Promote CliQ Purchases":
            next_state[2] = min(next_state[2] + 1, 2)

        elif action == "Encourage Money Transfers":
            next_state[3] = min(next_state[3] + 1, 2)

    elif system == "eFAWATEERcom":

        if action == "Attract New Users":
            next_state[0] = min(next_state[0] + 1, 2)

        elif action == "Onboard New Billers":
            next_state[1] = min(next_state[1] + 1, 2)

        elif action == "Move Cash Payments to Digital Channels":
            # Reduce cash payments
            next_state[2] = max(next_state[2] - 1, 0)
            # Increase digital payments
            next_state[3] = min(next_state[3] + 1, 2)

    elif system == "ACH":

        if action == "Promote JOD Transfers":
            next_state[0] = min(next_state[0] + 1, 2)

        elif action == "Promote USD Transfers":
            next_state[1] = min(next_state[1] + 1, 2)

        elif action == "Promote EUR and GBP Transfers":
            next_state[2] = min(next_state[2] + 1, 2)
            next_state[3] = min(next_state[3] + 1, 2)

    elif system == "ECCU":

        if action == "Improve Cheque Processing":
            next_state[0] = min(next_state[0] + 1, 2)

        elif action == "Reduce Returned-Cheque Count":
            next_state[1] = max(next_state[1] - 1, 0)

        elif action == "Reduce Returned-Cheque Risk Rates":
            next_state[2] = max(next_state[2] - 1, 0)
            next_state[3] = max(next_state[3] - 1, 0)

    return tuple(next_state)

In [26]:
def apply_action_with_budget(system, state, action, budget):

    action_cost = system_actions[system][action]
    # Check if budget is enough
    if action_cost > budget:
        return state, budget, action_cost, False

    next_state = apply_action(system,state,action)
    remaining_budget = budget - action_cost

    return (next_state,remaining_budget,action_cost,True)

In [27]:
positive_features = {
    "JoMoPay": [0, 1, 2],
    "CliQ": [0, 1, 2, 3],
    "eFAWATEERcom": [0, 1, 3],
    "ACH": [0, 1, 2, 3],
    "ECCU": [0]
}
negative_features = {
    "JoMoPay": [3],
    "CliQ": [],
    "eFAWATEERcom": [2],
    "ACH": [],
    "ECCU": [1, 2, 3]
}

In [28]:
def calculate_state_score(system, state):
    score = 0
    for index in positive_features[system]:
        score += state[index] #Positive --> high
    for index in negative_features[system]:
        score += 2 - state[index] #Negative --> low
    return score

In [29]:
def is_goal_state(system, state):
    return calculate_state_score(system, state) == 8

In [30]:
improvement_weight = 10
cost_penalty = 0.05
step_penalty = 0.5
ineffective_penalty = 2

In [31]:
def calculate_reward(system, current_state, next_state, action_cost):

    current_score = calculate_state_score(system, current_state)
    next_score = calculate_state_score(system, next_state)

    improvement = next_score - current_score

    # Encourage operational improvement
    reward = improvement * improvement_weight

    # Penalise excessive intervention cost
    reward -= action_cost * cost_penalty

    # Penalise unnecessary steps
    reward -= step_penalty

    # Penalise ineffective actions
    if next_state == current_state:
        reward -= ineffective_penalty

    return reward

In [32]:
def check_episode_end(system,state,budget,action,step,max_steps):

    if is_goal_state(system, state):
        return True, "Goal state reached"

    if action == "Do Nothing":
        return True, "No intervention selected"

    intervention_costs = []
    for action_name, cost in system_actions[system].items():
        if action_name != "Do Nothing":
            intervention_costs.append(cost)
    # Exclude Do Nothing because its cost is zero
    if budget < min(intervention_costs):
        return True, "Insufficient budget"

    if step >= max_steps:
        return True, "Maximum steps reached"

    return False, "Continue"

In [33]:
# All possible combinations of four feature levels
all_states = list(itertools.product([0, 1, 2], repeat=4))
state_to_index = {state: index for index, state in enumerate(all_states)}

In [34]:
action_lists = {}

for system in system_features:
    action_lists[system] = list(system_actions[system].keys())

In [35]:
config_1 = {
    "name": "Policy 1",
    "episodes": 3000,
    "alpha": 0.05,
    "gamma": 0.9,
    "epsilon": 1.0,
    "epsilon_decay": 0.97,
    "epsilon_min": 0.05
}

config_2 = {
    "name": "Policy 2",
    "episodes": 5000,
    "alpha": 0.05,
    "gamma": 0.9,
    "epsilon": 1.0,
    "epsilon_decay": 0.99,
    "epsilon_min": 0.05
}

max_steps = 5

training_budgets = [25, 50, 100]
evaluation_budgets = [25, 50, 100]

In [41]:
def train_policy(config):

    random.seed(42)
    np.random.seed(42)

    policy_q_tables = {}

    # Create Q-table for each payment system
    for system in system_features:
        policy_q_tables[system] = np.zeros((len(all_states), len(action_lists[system])))

    training_rewards = {}
    config_success = []
    config_steps = []
    config_remaining_budget = []

    for system in system_features:

        print("\nTraining", config["name"], "-", system)

        current_epsilon = config["epsilon"]
        episode_rewards = []

        for episode in range(config["episodes"]):

            current_state = random.choice(all_states)
            initial_state = current_state
            budget = random.choice(training_budgets)

            total_reward = 0
            steps_used = 0

            for step in range(1, max_steps + 1):
                if is_goal_state(system, current_state):
                    break

                steps_used = step

                # State index
                state_index = state_to_index[current_state]

                # Find actions that can be afforded
                affordable_action_indices = []

                for action_index, action in enumerate(action_lists[system]):

                    action_cost = system_actions[system][action]

                    if action_cost <= budget:
                        affordable_action_indices.append(action_index)

                # Epsilon-greedy action selection
                if random.random() < current_epsilon:
                    action_index = random.choice(affordable_action_indices)

                else:
                    action_index = max(affordable_action_indices,key=lambda index: policy_q_tables[system][state_index, index])

                action = action_lists[system][action_index]
                next_state, remaining_budget, action_cost, _ = (apply_action_with_budget(system,current_state,action,budget))
                reward = calculate_reward(system,current_state,next_state,action_cost)

                total_reward += reward
                # Next state index
                next_state_index = state_to_index[next_state]
                done, _  = check_episode_end(
                    system,
                    next_state,
                    remaining_budget,
                    action,
                    step,
                    max_steps
                )
                # Q-learning target
                if done:
                    target = reward
                else:
                    next_affordable_actions = []

                    for next_action_index, next_action in enumerate(action_lists[system]):
                        next_action_cost = system_actions[system][next_action]
                        if next_action_cost <= remaining_budget:
                            next_affordable_actions.append(next_action_index)
                    best_next_q = max(
                        policy_q_tables[system][next_state_index,next_action_index]
                        for next_action_index in next_affordable_actions
                    )

                    target = (reward + config["gamma"] * best_next_q)
                # Q-learning update
                old_q_value = policy_q_tables[system][state_index,action_index]
                policy_q_tables[system][state_index, action_index] = (old_q_value + config["alpha"] * (target - old_q_value))

                current_state = next_state
                budget = remaining_budget
                if done:
                    break
            episode_rewards.append(total_reward)
            initial_score = calculate_state_score(system,initial_state)
            final_score = calculate_state_score(system,current_state)

            # Success means operational improvement
            if final_score > initial_score:
                config_success.append(1)
            else:
                config_success.append(0)

            config_steps.append(steps_used)
            config_remaining_budget.append(budget)
            # Reduce epsilon
            current_epsilon = max(current_epsilon * config["epsilon_decay"],config["epsilon_min"])

        training_rewards[system] = episode_rewards
        average_reward = np.mean(episode_rewards[-500:])
        print("Average reward:",round(average_reward, 2))

    # Calculate average learned Q value
    q_values = []

    for system in policy_q_tables: q_values.extend(policy_q_tables[system].flatten())
    average_q_value = np.mean(q_values)
    average_training_reward = np.mean([np.mean(training_rewards[system][-500:]) for system in system_features])
    success_rate = np.mean(config_success)
    average_steps = np.mean(config_steps)
    average_remaining_budget = np.mean(config_remaining_budget)
    return (
        policy_q_tables,
        average_q_value,
        average_training_reward,
        success_rate,
        average_steps,
        average_remaining_budget
    )

In [42]:
(policy_1_q_tables,policy_1_average_q,policy_1_average_reward,policy_1_success,policy_1_steps,policy_1_remaining_budget) = train_policy(config_1)
(policy_2_q_tables,policy_2_average_q,policy_2_average_reward,policy_2_success,policy_2_steps,policy_2_remaining_budget) = train_policy(config_2)


Training Policy 1 - JoMoPay
Average reward: 13.31

Training Policy 1 - CliQ
Average reward: 15.57

Training Policy 1 - eFAWATEERcom
Average reward: 12.65

Training Policy 1 - ACH
Average reward: 12.44

Training Policy 1 - ECCU
Average reward: 11.32

Training Policy 2 - JoMoPay
Average reward: 13.04

Training Policy 2 - CliQ
Average reward: 15.94

Training Policy 2 - eFAWATEERcom
Average reward: 13.15

Training Policy 2 - ACH
Average reward: 12.22

Training Policy 2 - ECCU
Average reward: 13.15


In [43]:
comparison = pd.DataFrame({

    "Configuration": ["Policy 1","Policy 2"],
    "Average Q Value": [round(policy_1_average_q, 3),round(policy_2_average_q, 3)],
    "Average Training Reward": [round(policy_1_average_reward, 3),round(policy_2_average_reward, 3)],
    "Success Rate": [round(policy_1_success, 3),round(policy_2_success, 3)],
    "Average Steps": [round(policy_1_steps, 3),round(policy_2_steps, 3)],
    "Average Remaining Budget": [round(policy_1_remaining_budget,3),round(policy_2_remaining_budget,3)]
})

comparison

,Configuration,Average Q Value,Average Training Reward,Success Rate,Average Steps,Average Remaining Budget
0,Policy 1,2.971,13.058,0.858,1.695,14.485
1,Policy 2,3.612,13.502,0.866,1.718,14.067


In [44]:
if policy_2_success > policy_1_success:

    best_q_tables = policy_2_q_tables
    best_policy_name = "Policy 2"
    best_config = config_2

else:

    best_q_tables = policy_1_q_tables
    best_policy_name = "Policy 1"
    best_config = config_1

print("Selected Best Policy:",best_policy_name)

Selected Best Policy: Policy 2


In [48]:
saved_policy = {
    "policy_name": best_policy_name,
    "q_tables": best_q_tables,
    "config": best_config,
    "reward_parameters": {
        "improvement_weight": improvement_weight,
        "cost_penalty": cost_penalty,
        "step_penalty": step_penalty,
        "ineffective_penalty": ineffective_penalty
    },

    "system_features": system_features,
    "system_bins": system_bins,
    "system_actions": system_actions,
    "action_lists": action_lists,
    "positive_features": positive_features,
    "negative_features": negative_features,
    "all_states": all_states,
    "state_to_index": state_to_index,
    "max_steps": max_steps,

    "comparison": comparison.to_dict(orient="records"),

    "simulation_note": (
        "Intervention outcomes, costs, and state transitions "
        "are simulated because the dataset does not contain "
        "actual intervention records."
    )
}

os.makedirs("pkl_files", exist_ok=True)
joblib.dump(saved_policy, "pkl_files/best_rl_policy.pkl")

print("\nBest policy saved as best_rl_policy.pkl")


Best policy saved as best_rl_policy.pkl
